# Prover-Verifier Deliberation (PVD)

A **Prover** LLM answers a question and defends its reasoning; a **Verifier** LLM challenges sub-claims until it Accepts, Rejects, or hits the fatigue limit.

The key metric is **ANC (Accept + No Change)**: questions where the verifier Accepts *and* the prover never changed its answer. This subset has ~93% accuracy on GPQA Diamond at ~55% coverage.

**Run from the repo root with the `trust-but-verify` conda environment active.**

## 1. Setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env in the repo root

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (required for GPQA)"
print("API keys loaded.")

In [ ]:
from rttr.common import load_gpqa_diamond, format_mcq, load_prompt, cost_usd, DATA_DIR
from rttr.pvd import run_question, run_async
from rttr import SCHEMA_VERSION

## 2. Configure

In [ ]:
PROVER_MODEL   = "claude-sonnet-4-6"
VERIFIER_MODEL = "claude-haiku-4-5-20251001"  # change to PROVER_MODEL for self-play

FATIGUE_LIMIT  = 12   # max challenge rounds per attempt
MAX_ATTEMPTS   = 1    # set to 5 for pvd_retry (restarts on Reject)
MIN_CHALLENGES = 0    # set to 1 to force at least one verifier challenge

N_QUESTIONS    = 5    # how many GPQA questions to run in the batch
SEED           = 42   # reproducibility (matches paper)

## 3. Single-question demo

Run PVD on one GPQA Diamond question and inspect the full transcript.

In [ ]:
items = load_gpqa_diamond(n=1, seed=SEED)
item  = items[0]

print(f"Domain: {item.domain} / {item.subdomain}")
print()
print(format_mcq(item))
print(f"\nCorrect answer: {item.correct_letter}")

In [ ]:
# Run PVD — this makes live API calls; watch the output stream below.
# `await` works natively in Jupyter (IPykernel >= 6).

verifier_sys = load_prompt(
    "pvd_verifier_system_min1" if MIN_CHALLENGES > 0 else "pvd_verifier_system"
)

result = await run_question(
    item=item,
    qnum=1,
    total=1,
    prover_model=PROVER_MODEL,
    verifier_model=VERIFIER_MODEL,
    verifier_system_prompt=verifier_sys,
    thinking_budget=0,
    fatigue_limit=FATIGUE_LIMIT,
    max_attempts=MAX_ATTEMPTS,
    min_challenges=MIN_CHALLENGES,
)

In [ ]:
def show_transcript(result):
    """Print the deliberation transcript in a readable format."""
    ansi = lambda s: s  # keep ANSI stripped for notebooks

    for attempt in result["attempts"]:
        print(f"{'='*60}")
        print(f"Attempt {attempt['attempt_num']}  outcome={attempt['outcome']}")
        print(f"{'='*60}")
        for turn in attempt["transcript"]:
            if turn["role"] == "prover":
                print(f"\n[PROVER – turn {turn['turn']}]  answer={turn['answer']}")
                print(f"  Statement : {turn['statement'][:400]}")
                if turn.get("subclaims"):
                    for i, sc in enumerate(turn["subclaims"][:3], 1):
                        print(f"  Sub-claim {i}: {str(sc)[:200]}")
            else:
                verdict = turn["verdict"]
                marker  = {"Accept": "✓", "Reject": "✗", "Challenge": "?"}.get(verdict, "·")
                print(f"\n[VERIFIER – turn {turn['turn']}]  {marker} {verdict}")
                if turn.get("challenged_claim"):
                    print(f"  Challenged: {str(turn['challenged_claim'])[:200]}")
                if turn.get("challenge"):
                    print(f"  Challenge : {turn['challenge'][:400]}")
                if turn.get("reasoning"):
                    print(f"  Reasoning : {turn['reasoning'][:200]}")

    print(f"\n{'='*60}")
    correct_str = "CORRECT ✓" if result["correct"] else "WRONG ✗"
    print(f"Result   : {correct_str}  (correct={result['correct_letter']}, prover={result['prover_answer']})")
    print(f"Outcome  : {result['outcome']}")
    print(f"Attempts : {result['num_attempts']}  Rounds: {result['total_rounds']}")
    print(f"Cost     : ${result['cost_usd']:.4f}")


show_transcript(result)

## 4. Batch run on GPQA Diamond

Runs `N_QUESTIONS` questions concurrently and saves results to `data/gpqa_results_pvd_notebook.json`.  
The run resumes automatically if interrupted.

In [ ]:
output_path = str(DATA_DIR / "gpqa_results_pvd_notebook.json")

config = {
    "run_key": "pvd_notebook",
    "protocol": "pvd",
    "dataset": {"name": "gpqa_diamond", "n": N_QUESTIONS, "seed": SEED},
    "pvd": {
        "prover_model":           PROVER_MODEL,
        "verifier_model":         VERIFIER_MODEL,
        "thinking_budget_tokens": 0,
        "fatigue_limit":          FATIGUE_LIMIT,
        "max_attempts":           MAX_ATTEMPTS,
        "min_challenges":         MIN_CHALLENGES,
        "self_play":              PROVER_MODEL == VERIFIER_MODEL,
    },
    "logging":  {"verbose": True, "output": output_path, "_output_abs": output_path},
    "compute":  {"concurrent": 3},
    "_config_path": "(notebook)",
}

await run_async(config)

## 5. Inspect results

In [ ]:
with open(output_path) as f:
    raw = json.load(f)

records = [r for r in raw if isinstance(r, dict) and not r.get("_meta")]
n       = len(records)

correct  = sum(r["correct"] for r in records)
anc      = [r for r in records if r["outcome"] == "accept_no_change"]
non_anc  = [r for r in records if r["outcome"] != "accept_no_change"]

hc_prec    = 100 * sum(r["correct"] for r in anc)     / len(anc)     if anc     else float("nan")
non_hc_acc = 100 * sum(r["correct"] for r in non_anc) / len(non_anc) if non_anc else float("nan")
total_cost = sum(r["cost_usd"] for r in records)

print(f"Questions : {n}")
print(f"Accuracy  : {correct}/{n} = {100*correct/n:.1f}%")
print(f"ANC cov   : {len(anc)}/{n} = {100*len(anc)/n:.1f}%")
print(f"ANC prec  : {hc_prec:.1f}%")
print(f"Non-ANC   : {non_hc_acc:.1f}%")
print(f"Gap       : {hc_prec - non_hc_acc:+.1f}pp")
print(f"Total cost: ${total_cost:.4f}  (${total_cost/n:.4f}/question)")
print()

# Per-question summary table
print(f"{'Q':>3}  {'Domain':<25} {'Ans':>3} {'Cor':>3} {'Outcome':<20} {'Rnds':>4} {'Cost':>7}")
print("-" * 75)
for r in records:
    mark = "✓" if r["correct"] else "✗"
    dom  = r["domain"][:24]
    print(f"{r['question_num']:>3}  {dom:<25} {r['prover_answer']:>3} "
          f"{mark:>3} {r['outcome']:<20} {r['total_rounds']:>4} "
          f"${r['cost_usd']:>6.4f}")